# Dataset generation for Electricity Price Forecasting

This notebook generates the dataset for electricity price forecasting by:
1. Fetching data from ENTSOE API (2023-02-01 to 2026-02-01)
2. Including original exogenous variables (from paper Jesus Lago: FR Generation Forecast, FR Load Forecast)
3. Adding additional potentially useful variables
4. Validating variables using correlation and mutual information

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

from data.dataset_generator import ENTSOEDatasetGenerator
import pandas as pd

# Initialize generator
generator = ENTSOEDatasetGenerator()

# fetch the dataset (2023-02-01 to 2026-02-01)
df = generator.fetch_full_dataset(
    start_date="2023-02-01",
    end_date="2026-02-01",
    verbose=True
)


Fetching dataset from 2023-02-01 to 2026-02-01

Fetching data from 2023-02-01 to 2023-12-31


## Add time-based and derived features


In [2]:
# Add time-based features
print("Adding time-based features...")
df = generator.add_time_features(df)

# Add derived features
print("Adding derived features...")
df = generator.add_derived_features(df)

print(f"\nDataset shape after feature engineering: {df.shape}")
print(f"Columns: {list(df.columns)}")

Adding time-based features...
Adding derived features...

Dataset shape after feature engineering: (35166, 31)
Columns: ['Prices', 'FR_Generation_Forecast', 'FR_Load_Forecast', 'BE_Load_Forecast', 'BE_Load_Actual', 'BE_Generation_Forecast', 'BE_Generation_Actual', 'BE_Wind_Forecast', 'BE_Solar_Forecast', 'FR_Prices', 'NL_Prices', 'Flow_BE_FR', 'Flow_BE_NL', 'Flow_BE_DE', 'Hour', 'DayOfWeek', 'Month', 'IsWeekend', 'DayOfYear', 'WeekOfYear', 'Hour_sin', 'Hour_cos', 'DayOfWeek_sin', 'DayOfWeek_cos', 'Month_sin', 'Month_cos', 'BE_Load_Imbalance', 'BE_Generation_Imbalance', 'BE_Net_Position', 'Price_Spread_FR', 'Price_Spread_NL']


## Validate exogenous variables

In [3]:
# Validate variables using correlation and mutual information
validation_results = generator.validate_variables(
    df,
    train_split_date="2023-02-01",
    top_n=20
)

# Display top variables
print("\nTop variables by correlation and mutual information:")
print(validation_results.head(15))


Validating Exogenous Variables

Using training period: 2023-02-01 00:00:00 to 2025-07-31 23:00:00
Training samples: 21888

  Skipping BE_Generation_Forecast: 63.4% missing
  Skipping BE_Generation_Imbalance: 63.4% missing
  Skipping BE_Net_Position: 63.4% missing

Analyzing 27 variables

Variable                                  Correlation  Mutual Info  Missing %
--------------------------------------------------------------------------------
NL_Prices                                      0.9414       0.3756       0.0%
FR_Prices                                      0.8864       0.3440       0.0%
FR_Generation_Forecast                         0.5111       0.1465      40.2%
BE_Wind_Forecast                              -0.4346       0.0736       0.0%
FR_Load_Forecast                               0.4011       0.0625       0.0%
BE_Load_Actual                                 0.3613       0.0666       0.0%
BE_Load_Forecast                               0.3515       0.0671       0.0%
BE_So

## Select variables and save final dataset

In [4]:
# Select the best variables (exogenous variables with high correlation/MI) (excluding time features which are always picked)
time_features = [
    'Hour', 'DayOfWeek', 'Month', 'IsWeekend', 'DayOfYear', 'WeekOfYear',
    'Hour_sin', 'Hour_cos', 'DayOfWeek_sin', 'DayOfWeek_cos',
    'Month_sin', 'Month_cos'
]

# Get best exogenous variables
top_exog = validation_results[
    ~validation_results['Variable'].isin(time_features + ['Prices'])
].head(15)['Variable'].tolist()

# Combine Prices + time features + top exogenous variables
selected_vars = ['Prices'] + time_features + top_exog

print(f"\nSelected {len(selected_vars)} variables:")
print(f"  - Target: Prices")
print(f"  - Time features: {len(time_features)}")
print(f"  - Exogenous variables: {len(top_exog)}")
print(f"\nExogenous variables:")
for var in top_exog:
    print(f"  - {var}")

# Save datasets
from pathlib import Path

output_dir = Path.cwd().parent / "data" / "datasets"
output_dir.mkdir(parents=True, exist_ok=True)

# Save full dataset
generator.save_dataset(
    df,
    output_dir / "BE_ENTSOE_FULL.csv",
    selected_variables=None  # Save all variables
)

# Save selected variables dataset
generator.save_dataset(
    df,
    output_dir / "BE_ENTSOE.csv",
    selected_variables=selected_vars
)

# Save validation results
validation_results.to_csv(
    output_dir / "variable_validation.csv",
    index=False
)

print("\n" + "="*80)
print("Dataset generation complete!")
print("="*80)
print(f"\nFiles created:")
print(f"  1. {output_dir / 'BE_ENTSOE_FULL.csv'} - All variables")
print(f"  2. {output_dir / 'BE_ENTSOE.csv'} - Selected variables (for refactored code)")
print(f"  3. {output_dir / 'variable_validation.csv'} - Validation metrics")


Selected 28 variables:
  - Target: Prices
  - Time features: 12
  - Exogenous variables: 15

Exogenous variables:
  - NL_Prices
  - FR_Prices
  - FR_Generation_Forecast
  - BE_Wind_Forecast
  - FR_Load_Forecast
  - BE_Load_Actual
  - BE_Load_Forecast
  - BE_Solar_Forecast
  - Price_Spread_FR
  - Flow_BE_NL
  - BE_Generation_Actual
  - Flow_BE_FR
  - BE_Load_Imbalance
  - Price_Spread_NL
  - Flow_BE_DE

Dataset saved successfully!
  Path: /home/d1ff1cult/masterproef_new/data/datasets/BE_ENTSOE_FULL.csv
  Shape: (35166, 32)
  Variables: 31 (excluding Date)
  Date range: 2023-02-01 00:00:00 to 2026-01-31 23:45:00


Dataset saved successfully!
  Path: /home/d1ff1cult/masterproef_new/data/datasets/BE_ENTSOE.csv
  Shape: (35166, 29)
  Variables: 28 (excluding Date)
  Date range: 2023-02-01 00:00:00 to 2026-01-31 23:45:00


Dataset generation complete!

Files created:
  1. /home/d1ff1cult/masterproef_new/data/datasets/BE_ENTSOE_FULL.csv - All variables
  2. /home/d1ff1cult/masterproef_new/da